# 实践经验总结

## 1. 清晰的描述


In [ ]:
from langchain_core.tools import tool


# ✅ 正确示例
@tool(parse_docstring=True)
def search_flights(origin: str, destination: str, date: str) -> str:
    """
    搜索航班信息
    Args:
        origin: 出发城市，如"北京"
        destination: 目的地城市，如"上海"
        date: 出发日期，格式 YYYY‑MM‑DD
    Returns:
        可用航班的 JSON 列表
    """

## 2. 功能单一

In [ ]:
# ❌ 不好：一个工具做太多事
@tool
def do_everything(action: str, data: str) -> str:
    """做各种事情"""
    if action == "weather": ...
    elif action == "calculate": ...
    elif action == "search": ...

# ✅ 好：每个工具做一件事
@tool
def get_weather(city: str) -> str:
    """获取天气"""
    ...

@tool
def calculator(operation: str, a: float, b: float) -> str:
    """计算"""
    ...

## 3. 如何处理工具失败？

三层防护：

第1层：工具内部处理


In [ ]:
@tool
def divide(a: float, b: float) -> str:
    """
    除法计算
    Args:
        a: 被除数
        b: 除数
    """
    try:
        if b == 0:
            return "错误：除数不能为零"
        result = a / b
        return f"{a} / {b} = {result}"
    except Exception as e:
        return f"计算错误：{e}"

第2层：Agent 级重试（使用 prompt）

In [ ]:
from langchain.agents import create_agent
model = ""
agent = create_agent(
    model=model,
    tools=[...],
    prompt="如果工具失败，尝试使用其他方法解决问题。"
)

第3层：调用级重试

在大模型应用（如 LangChain）中，网络请求和外部工具调用是最容易掉链子的地方。@retry 就像是一个容错保险。


In [ ]:
from tenacity import retry, stop_after_attempt

# 1. 配置重试规则：如果失败，最多尝试 3 次（即第 1 次正常调用 + 2 次重试）
@retry(stop=stop_after_attempt(3))
def call_agent(question):
    # 2. 核心业务逻辑：调用 LangChain 的 Agent
    return agent.invoke({"messages": [{"role": "user", "content": question}]})

它的工作流程：

① 你调用 call_agent("你好")。

② 程序进入函数，执行 agent.invoke(...)。

③ 如果执行成功：正常返回结果， @retry 什么都不做。

④ 如果执行失败（报错）： @retry 会拦截这个错误，不让程序直接崩溃。它会默默地帮你再次触发agent.invoke(...)。

⑤ 如果连续 3 次都报错：它终于放弃了，把第 3 次的报错真正抛出来，程序此时才会报错中止。


## 4. 返回字符串

In [ ]:
import json
# ✅ 好：返回字符串
@tool
def get_user_info(user_id: str) -> str:
    """获取用户信息"""
    user = {"id": user_id, "name": "张三"}
    return json.dumps(user, ensure_ascii=False)  # 转成 JSON 字符串

# ❌ 不好：返回字典（某些情况可能有问题）
@tool
def get_user_info(user_id: str) -> dict:
    """获取用户信息"""
    return {"id": user_id, "name": "张三"}

在编写传统的Python代码时，返回字典（dict）显然更方便后续代码处理。但在LangChain的工具（Tools）生态中，强烈建议工具返回字符串（str）。因为：

1）大模型（LLM）的本质只吃“文本”

2）避免大模型“胡思乱想”（乱码与格式问题）

如果你返回一个包含中文的字典 {"name": "张三"} ，LangChain 在强制将其转换为字符串时，默认可能会采用 Unicode 编码，变成 {"name": "\u5f20\u4e09"}。大模型虽然能理解Unicode，但极易受到干扰。直接看到中文 张三 的大模型，和看到
\u5f20\u4e09的大模型，其输出的稳定性和准确率是有差距的。通过手动 json.dumps(..., ensure_ascii=False) ，你确保了喂给大模型的是最干净、最直观的纯文本。

json.dumps()是Python标准库json模块中的函数，用于将Python对象（如字典、列表）序列化成一个JSON格式的字符串。

ensure_ascii=False : 这个参数默认值为True，表示所有非ASCII字符（如中文）会被转换成 \uXXXX形式的转义序列。设置为 False后，中文、表情符号等字符就能在 JSON 字符串中正常显示，而不是一堆乱码。


## 5. 选择同步 vs 异步

同步工具 ：简单场景，CPU 密集型任务

异步工具 ：IO 密集型（API 调用、数据库、文件操作）


In [ ]:
# 同步
@tool
def sync_tool(x: str) -> str:
    return process(x)

# 异步
@tool
async def async_tool(x: str) -> str:
    return await async_process(x)